# To-do list
## Testing NSE histories   
[x] One Scrip blocking for Equity, Index and Option Histories   
[ ] Async One Scrip, One period   
[ ] Async One Scrip, Many period chunks   
[ ] Async All Scrips, All periods - with throttles and tqdm   
[ ] Standardizing history outputs   
[ ] Objectify   


In [1]:
import datetime
import logging

import pandas as pd
import requests

from typing import TypeAlias
JSON: TypeAlias = dict[str, "JSON"] | list["JSON"] | str | int | float | bool | None

# from nse import nse fetch #!!! Transfer code to nse.py after tests

def fetch(payload:str) -> JSON: 

    headers = {'User-Agent': 
               'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:74.0) Gecko/20100101 Firefox/74.0'}

    try:
        output = requests.get(payload,headers=headers).json()
    except ValueError:
        s = requests.Session()
        output = s.get("http://nseindia.com",headers=headers)
        output = s.get(payload,headers=headers).json()
    return output



# One scrip - blocking test

In [2]:
# Raw history IO
raw_eq_hist_io = 'https://www.nseindia.com/api/historical/cm/equity?symbol=SBIN&series=[%22EQ%22]&from=02-03-2023&to=21-04-2023'
raw_opt_eq_hist_io = 'https://www.nseindia.com/api/historical/fo/derivatives?&from=02-03-2023&to=21-04-2023&optionType=PE&strikePrice=500.00&expiryDate=27-Apr-2023&instrumentType=OPTSTK&symbol=SBIN'
raw_opt_index_hist_io = "https://www.nseindia.com/api/historical/fo/derivatives?symbol=NIFTY"


## One scrip One URL build

In [3]:
# Parameters
eq_symbol = 'SBIN'
idx_symbol = 'NIFTY'
days = 100
max_records = 35

end_date = datetime.datetime.now().date()
start_date = (end_date-datetime.timedelta(days=days))


In [4]:
# Build an equity URL

series = '[%22EQ%22]'
eq_base_url = "https://www.nseindia.com/api/historical/cm/equity"
eq_params = {'symbol': eq_symbol, 'series': series,
             'from': start_date.strftime('%d-%m-%Y'), 'to': end_date.strftime('%d-%m-%Y')}



In [5]:
def build_url(base_url: str, params: dict) -> str:
    """Build url from base with query parameters. Preserves `[]`"""
    encoded_params = '&'.join([f"{k}={v}" for k, v in params.items()])
    url = f"{base_url}?{encoded_params}"
    return url

url = build_url(eq_base_url, eq_params)
payload = fetch(url)
df = pd.DataFrame.from_records(payload['data'])
df.head()


,_id,CH_SYMBOL,CH_SERIES,CH_MARKET_TYPE,CH_TRADE_HIGH_PRICE,CH_TRADE_LOW_PRICE,CH_OPENING_PRICE,CH_CLOSING_PRICE,CH_LAST_TRADED_PRICE,CH_PREVIOUS_CLS_PRICE,...,CH_ISIN,CH_TIMESTAMP,TIMESTAMP,createdAt,updatedAt,__v,SLBMH_TOT_VAL,VWAP,mTIMESTAMP,CA
0,6495896228d6ae000711888e,SBIN,EQ,N,562.10,553.80,562.00,554.60,555.45,562.95,...,INE062A01020,2023-06-23,2023-06-22T18:30:00.000Z,2023-06-23T12:00:34.623Z,2023-06-23T12:00:34.623Z,0,None,556.89,23-Jun-2023,NaN
1,649437e05c29720007f6722e,SBIN,EQ,N,569.00,561.05,566.35,562.95,562.90,566.35,...,INE062A01020,2023-06-22,2023-06-21T18:30:00.000Z,2023-06-22T12:00:32.545Z,2023-06-22T12:00:32.545Z,0,None,564.62,22-Jun-2023,NaN
2,6492e67eeff1a80007e01784,SBIN,EQ,N,569.50,565.65,567.40,566.35,566.30,567.40,...,INE062A01020,2023-06-21,2023-06-20T18:30:00.000Z,2023-06-21T12:01:02.772Z,2023-06-21T12:01:02.772Z,0,None,567.46,21-Jun-2023,NaN
3,649194e0cba4890007b6915a,SBIN,EQ,N,569.45,562.55,568.85,567.40,568.00,568.85,...,INE062A01020,2023-06-20,2023-06-19T18:30:00.000Z,2023-06-20T12:00:32.605Z,2023-06-20T12:00:32.605Z,0,None,565.75,20-Jun-2023,NaN
4,6490437ee2783a00078bad43,SBIN,EQ,N,572.75,565.90,571.25,568.85,569.20,571.25,...,INE062A01020,2023-06-19,2023-06-18T18:30:00.000Z,2023-06-19T12:01:02.673Z,2023-06-19T12:01:02.673Z,0,None,568.29,19-Jun-2023,NaN


## One scrip - many URLs generation (history dates split)

### For Equity

In [6]:
# Inputs
eq_symbol = 'SBIN'
idx_symbol = 'NIFTY'
days = 80
max_records = 35
end_date = datetime.datetime.now().date()
nse_style = True # converts date friendly to nse.com

In [7]:
def hist_date_splits(symbol: str,
                     end_date: datetime.date,
                     days: int = 100, 
                     max_records: int = 35, 
                     nse_style: bool=True):
    
    """Generates date splits"""

    # Computations
    start_date = end_date - datetime.timedelta(days)

    # ..bin the dates
    periods = int((end_date-start_date).days/max_records)+1
    ranged = pd.date_range(start = start_date, end=end_date, periods=periods)

    if nse_style:
        ranged = ranged.strftime('%d-%m-%Y')
    else:
        ranged = ranged.date

    z_bins = zip(ranged[:-1], ranged[1:])

    return z_bins

hist_dates = hist_date_splits(symbol=eq_symbol, end_date=end_date, days=days, max_records=max_records, nse_style=True)

In [8]:
eq_base_url = "https://www.nseindia.com/api/historical/cm/equity"
eq_params = [{'symbol': eq_symbol,
        'series': '[%22EQ%22]',
        'from': start,
        'to': end} for start, end in list(hist_dates)]

In [9]:
# Generate the URLs
urls = [build_url(eq_base_url, param) for param in eq_params]
# urls


### Async - history for one scrip URL containing the date range

In [10]:
# Async one-scrip with aiohttp
import aiohttp

In [11]:
import requests

async def async_history(url: str, timeout_seconds: float=1) -> pd.DataFrame:

    agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:74.0) Gecko/20100101 Firefox/74.0'
    headers = {'User-Agent': agent}

    timeout = aiohttp.ClientTimeout(total = timeout_seconds)

    async with aiohttp.ClientSession() as session:

        try:
            async with session.get(url, headers=headers, timeout=timeout) as resp: 
                result = await resp.json()
                # print(f"\n result through first session\n{result}\n")

        except aiohttp.ContentTypeError as e: # for content type error use blocking requests

            try:
                s = requests.Session()
                result = s.get("http://nseindia.com",headers=headers)
                result = s.get(url, headers=headers, timeout=timeout.total).json()
                # print(f"\n result through second request session\n{result}\n")

            except Exception as e:
                print(f"Error in {e}")
                logging.error(f"Error in {e}")

        return result

# data = await async_history(url=url)


In [25]:
import asyncio
import tqdm

async def async_histories(urls: list, timeout_seconds: float=2):

    results = []

    for url in tqdm.tqdm(urls):
        task = asyncio.create_task(async_history(url, timeout_seconds), name=url.split("?")[1].replace("series=[%22EQ%22]&", ""))
        await task
        results.append(task)

    return results

# Generate the URLs
end_date = datetime.datetime.now().date()
days = 1000
max_records = 35
hist_dates = hist_date_splits(symbol=eq_symbol, end_date=end_date, days=days, max_records=max_records, nse_style=True)

eq_base_url = "https://www.nseindia.com/api/historical/cm/equity"
eq_params = [{'symbol': eq_symbol,
        'series': '[%22EQ%22]',
        'from': start,
        'to': end} for start, end in list(hist_dates)]
urls = [build_url(eq_base_url, param) for param in eq_params]

data = await async_histories(urls=urls)

  4%|▎         | 1/28 [00:00<00:04,  5.51it/s]

100%|██████████| 28/28 [00:05<00:00,  5.40it/s]


In [26]:
dfs = []
for d in data:
    try:
        dfs.append(pd.DataFrame.from_records(d.result()['data']))
    except Exception as e:
        logging.error(f"Error {e}")

In [27]:
df = pd.concat(dfs, ignore_index=True)
df

,_id,CH_SYMBOL,CH_SERIES,CH_MARKET_TYPE,CH_TRADE_HIGH_PRICE,CH_TRADE_LOW_PRICE,CH_OPENING_PRICE,CH_CLOSING_PRICE,CH_LAST_TRADED_PRICE,CH_PREVIOUS_CLS_PRICE,...,CH_ISIN,CH_TIMESTAMP,TIMESTAMP,createdAt,updatedAt,__v,SLBMH_TOT_VAL,VWAP,mTIMESTAMP,CA
0,60c1fca76804760008d04d3c,SBIN,EQ,N,197.25,190.05,192.20,196.05,196.00,189.25,...,INE062A01020,2020-11-02,2020-11-01T18:30:00.000Z,2021-06-10T11:51:03.337Z,2021-06-10T11:51:03.337Z,0,None,194.16,02-Nov-2020,NaN
1,60c1fc927ad7b20008233ce5,SBIN,EQ,N,192.00,186.15,189.35,189.25,189.90,188.70,...,INE062A01020,2020-10-30,2020-10-29T18:30:00.000Z,2021-06-10T11:50:42.084Z,2021-06-10T11:50:42.084Z,0,None,189.11,30-Oct-2020,NaN
2,60c1fc7d57283a0008f11cc9,SBIN,EQ,N,190.70,185.90,189.35,188.70,189.30,190.45,...,INE062A01020,2020-10-29,2020-10-28T18:30:00.000Z,2021-06-10T11:50:21.560Z,2021-06-10T11:50:21.560Z,0,None,188.17,29-Oct-2020,NaN
3,60c1fc68f9fddb000867607f,SBIN,EQ,N,195.00,189.05,195.00,190.45,190.50,194.65,...,INE062A01020,2020-10-28,2020-10-27T18:30:00.000Z,2021-06-10T11:50:00.960Z,2021-06-10T11:50:00.960Z,0,None,191.54,28-Oct-2020,NaN
4,60c1fc537ad7b200082333fb,SBIN,EQ,N,197.55,192.25,197.25,194.65,194.45,196.70,...,INE062A01020,2020-10-27,2020-10-26T18:30:00.000Z,2021-06-10T11:49:39.689Z,2021-06-10T11:49:39.689Z,0,None,194.96,27-Oct-2020,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
694,64709f7f9ab47100073f077a,SBIN,EQ,N,587.75,580.55,581.25,586.00,585.75,581.25,...,INE062A01020,2023-05-26,2023-05-25T18:30:00.000Z,2023-05-26T12:01:03.036Z,2023-05-26T12:01:03.036Z,0,None,585.13,26-May-2023,NaN
695,646f4dfef4ee6800078f183a,SBIN,EQ,N,582.90,577.00,582.00,581.25,580.70,582.70,...,INE062A01020,2023-05-25,2023-05-24T18:30:00.000Z,2023-05-25T12:01:02.240Z,2023-05-25T12:01:02.240Z,0,None,580.06,25-May-2023,NaN
696,646dfc7f340f0d0007e5d3ff,SBIN,EQ,N,585.25,578.25,579.80,582.70,582.00,581.60,...,INE062A01020,2023-05-24,2023-05-23T18:30:00.000Z,2023-05-24T12:01:03.131Z,2023-05-24T12:01:03.131Z,0,None,583.06,24-May-2023,NaN
697,646caae138862f0007f842e6,SBIN,EQ,N,583.70,576.80,578.90,581.60,580.30,577.15,...,INE062A01020,2023-05-23,2023-05-22T18:30:00.000Z,2023-05-23T12:00:33.136Z,2023-05-23T12:00:33.136Z,0,None,580.66,23-May-2023,NaN


In [ ]:
df = pd.DataFrame.from_records(data['data'])
df.head()


In [ ]:
# "https://www.nseindia.com/api/historical/cm/equity?symbol=SBIN&series=[%22EQ%22]&from=06-04-2023&to=16-05-2023".split("?")[1].replace("series=[%22EQ%22]&", "")

In [ ]:
urls

In [ ]:
import asyncio
async def async_histories(urls: list, timeout: float=2.0):

    tasks = [asyncio.create_task(async_history(url=url, timeout=timeout), name=url.split("?")[1].replace("series=[%22EQ%22]&", "")) for url in urls]
    res = asyncio.gather(*tasks)

    return res

In [ ]:
result = await async_histories(urls)

In [ ]:
type(result)

In [ ]:

async def main():

    async with aiohttp.ClientSession() as session:
        async with session.get(url, headers=headers) as resp:

            # print(resp.status)
            # print(await resp.text())

            # await resp.text()

            try:
                result = await resp.json()
            
            except (ValueError, TypeError, aiohttp.ContentTypeError):

                async def req():

                    s = requests.Session()
                    res = s.get("http://nseindia.com",headers=headers)
                    res = s.get(url, headers=headers).json()

                    return res
                
                result = await req()
                print(result)

            # result = await resp.text()
            
    return result
            

result = await main()

In [ ]:
# import nest_asyncio
# nest_asyncio.apply()

url = urls[0]


In [ ]:
# Fetch a payload asynchronously

import aiohttp
import asyncio

timeout_seconds = 2

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:74.0) Gecko/20100101 Firefox/74.0'
}

async def fetch(session, url, timeout_seconds):
    async with session.get(url, timeout = timeout_seconds) as resp:
        try:
            assert resp.status == 200
        except AssertionError:
            logging.error(AssertionError)
            s = await aiohttp.ClientSession(headers=headers, timeout=timeout_seconds)
            print(s)

        return await resp.text()

async def main(url, timeout_seconds):
    async with aiohttp.ClientSession(headers=headers, timeout=timeout_seconds) as session:
        html = await fetch(session, url, timeout_seconds)
        print(html)

# loop = asyncio.get_event_loop()
# loop.run_until_complete(main())
# asyncio.run(main())
await main(url, timeout_seconds)

In [ ]:
# Fetch multiple payloads asynchronously

max_concurrent_requests = 2  # Maximum number of concurrent requests
timeout_seconds = 10  # Timeout for each individual request in seconds
throttle_delay = 1  # Delay between requests in seconds

async def fetch_with_throttling(client, url, throttle_delay):
    await asyncio.sleep(throttle_delay)  # Throttle requests
    return await fetch(client, url)

async def main(urls, timeout_seconds, max_concurrent_requests, throttle_delay):
    for url in urls:
        async with aiohttp.ClientSession(headers=headers, timeout=timeout_seconds) as session:
            semaphore = asyncio.Semaphore(max_concurrent_requests)
            async with semaphore:
                html = await fetch(session, url, timeout_seconds)
                print(html)        


In [ ]:
await main(urls, timeout_seconds, max_concurrent_requests, throttle_delay)

In [ ]:
import asyncio
import aiohttp
import time

MAX_CONCURRENT_REQUESTS = 2  # Maximum number of concurrent requests
REQUEST_TIMEOUT = 10  # Timeout for each individual request in seconds
THROTTLE_DELAY = 1  # Delay between requests in seconds

async def fetch(session, url):
    async with session.get(url, timeout=REQUEST_TIMEOUT) as response:
        return await response.text()

async def fetch_with_throttling(session, url):
    await asyncio.sleep(THROTTLE_DELAY)  # Throttle requests
    return await fetch(session, url)

async def main(urls):
    async with aiohttp.ClientSession() as session:
        tasks = []
        semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
        
        for url in urls:
            async with semaphore:
                task = asyncio.ensure_future(fetch_with_throttling(session, url))
                tasks.append(task)

        responses = await asyncio.gather(*tasks, return_exceptions=True)
        
        # Process the responses
        for url, response in zip(urls, responses):
            if isinstance(response, Exception):
                print(f"Error fetching URL {url}: {response}")
            else:
                print(f"Response from {url}: {response[:50]}...")
    
if __name__ == '__main__':
    start_time = time.time()
    loop = asyncio.get_event_loop()
    loop.run_until_complete(main(urls))
    elapsed_time = time.time() - start_time
    print(f"Total elapsed time: {elapsed_time:.2f} seconds")


In [ ]:
import asyncio
import aiohttp

coros = []

async def async_fetch(url):

    output = nsefetch(url)

    return output

my_timeout = aiohttp.ClientTimeout(
    total=None, # default value is 5 minutes, set to `None` for unlimited timeout
    sock_connect=4, # How long to wait before an open socket allowed to connect
    sock_read=4 # How long to wait with no data being read before timing out
)

client_args = dict(
    trust_env=True,
    timeout=my_timeout
)

# Pass args
async with aiohttp.ClientSession(**client_args) as session:
    ...
    # do stuff here
    for url in urls:
        coros.append(async_fetch(url))
    
    # Try awaiting, if a timeout, do something else
    try:
        values = await asyncio.gather(*coros, return_exceptions=True)
    except asyncio.exceptions.TimeoutError:
        print('Timeout!')